## Model refining process
Here‘s the process of how we developed our house price prediction model using the updated competition data: 1,200 training houses and 260 test houses. 


### Features used in our final model

**Original features.** 

The updated training file contains Id, SalePrice, and **79 housing features**. We started with all 79 features, leaving out Id because it only identifies each house and SalePrice because it is what we want to predict. 

Our final model uses all **79 original features**, listed here in the same order as the CSV. These are all from the provided dataset:

- 01–10: MSSubClass, MSZoning, LotFrontage, LotArea, Street, Alley, LotShape, LandContour, Utilities, LotConfig
- 11–20: LandSlope, Neighborhood, Condition1, Condition2, BldgType, HouseStyle, OverallQual, OverallCond, YearBuilt, YearRemodAdd
- 21–30: RoofStyle, RoofMatl, Exterior1st, Exterior2nd, MasVnrType, MasVnrArea, ExterQual, ExterCond, Foundation, BsmtQual
- 31–40: BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinSF1, BsmtFinType2, BsmtFinSF2, BsmtUnfSF, TotalBsmtSF, Heating, HeatingQC
- 41–50: CentralAir, Electrical, 1stFlrSF, 2ndFlrSF, LowQualFinSF, GrLivArea, BsmtFullBath, BsmtHalfBath, FullBath, HalfBath
- 51–60: BedroomAbvGr, KitchenAbvGr, KitchenQual, TotRmsAbvGrd, Functional, Fireplaces, FireplaceQu, GarageType, GarageYrBlt, GarageFinish
- 61–70: GarageCars, GarageArea, GarageQual, GarageCond, PavedDrive, WoodDeckSF, OpenPorchSF, EnclosedPorch, 3SsnPorch, ScreenPorch
- 71–79: PoolArea, PoolQC, Fence, MiscFeature, MiscVal, MoSold, YrSold, SaleType, SaleCondition

**Six features we created.** 

We used pandas to calculate six additional features from the provided columns.

1. **TotalArea = TotalBsmtSF + 1stFlrSF + 2ndFlrSF.** Reason: one number summarizes the basement and main floors, so the model can use an overall size signal rather than learning three separate effects only.
2. **TotalBathrooms = FullBath + 0.5 × HalfBath + BsmtFullBath + 0.5 × BsmtHalfBath.** Reason: a half bath offers less than a full bath; this gives one approximate measure of bathroom capacity while retaining the original columns too.
3. **HouseAge = YrSold − YearBuilt.** Reason: age when sold is more directly interpretable than construction year alone and may capture wear or differences between old and new homes.
4. **QualityLivingArea = OverallQual × capped GrLivArea.** Reason: extra living space may be worth more in a higher-quality home. GrLivArea is first capped at 4,000 square feet to reduce extreme linear extrapolation. This is a small interaction of the kind introduced by PolynomialFeatures.
5. **LargeEdwards = 1 if Neighborhood is Edwards and original GrLivArea exceeds 4,000; otherwise 0.** Reason: price patterns for unusually large homes may vary by neighborhood, so we tested a separate adjustment for this observable combination. Training support is limited (one matching row), so the estimated effect may overfit.
6. **QualityGarageCars = OverallQual × GarageCars.** Reason: a garage space may have a different price contribution in homes of different overall quality. We tested this single interaction after broader polynomial terms and other small interactions. Five-partition mean RMSE fell from 27,347.14 to 27,184.01, and Kaggle public RMSE fell from 27,643.51679 to 27,003.58788.

**Preprocessing.** We treated MSSubClass as a categorical variable, capped GrLivArea at 4,000, filled missing values, standardized numerical features, and one-hot encoded categorical features. The model therefore starts with **85 input columns** (79 original + 6 created), but one-hot encoding produces more columns internally. None of these features uses hidden test prices or another team's predictions.

### Models we submitted to Kaggle

| Submission | Kaggle File Name | Model | Public RMSE |
| --- | --- | --- | ---: | 
| Simple random forest | submission_04.csv | RandomForestRegressor | 30,697.63354 |
| Capped Ridge | submission_1101-05.csv | Ridge without LargeEdwards | 54,417.66090 |
| Previous best Ridge | submission_1101-6.csv | Ridge with LargeEdwards | 27,643.51679 | 
| Ridge plus forest average | submission_1102-1.csv | 75% previous Ridge, 25% forest | 27,802.62697 |
| **Final submission** | submission_1102-2.csv | One Ridge with QualityGarageCars | **27,003.58788** |

We did not record valid Kaggle public scores for the class baselines or the polynomial-only experiments, so their local validation results should not be mistaken for leaderboard scores. We also tried a separate advanced blend that scored 26,364.39873, but it used boosting beyond the course methods covered here. It is not the final model in this document.

### Models improving process

1. Starting with the class baseline

We first wanted to make sure we were using the updated data correctly, predicting the right target, and keeping the test rows in the correct order.

First try fit LinearRegression using only OverallQual. Its 20% holdout RMSE was **49,041.38**. Later we tried a model with 20 original predictors with missing-value filling and categorical encoding; its RMSE on the same holdout was **45,628.45**. We also checked an exploratory all-feature linear-regression pipeline on one five-fold partition; its out-of-fold RMSE was **35,388.55**. 

Using more features and handling missing values improved on the one-feature model. We also wanted a model that used more than one predictor. Since ordinary linear regression can be unstable when predictors are strongly related, we decided to try Ridge next.

2. Trying polynomial features

The relationship between price and features like size or quality may not be perfectly linear. We had learned how to add squared terms and interactions with PolynomialFeatures, so we tested whether they helped.

We added degree-two terms for five interpretable numeric predictors: OverallQual, GrLivArea, TotalBsmtSF, GarageCars, and YearBuilt. Ridge controlled the larger feature set. On three five-fold partitions, alpha=30 averaged **31,791.90** RMSE; alpha=100 averaged **31,856.24** and alpha=300 averaged **31,996.16**. A separate one-partition comparison that also included other engineered inputs gave **32,708.24**; it used a different feature set and is reported only as an additional exploratory check.

Adding many quadratic terms did not work as well as the simpler Ridge models we tried later. Instead of keeping all those terms, we focused on a few interactions that were easy to explain. We did not get a valid Kaggle score for a polynomial-only model.

3. Comparing with a random forest

 We wanted to compare the RandomForestRegressor with linear models, especially for unusually large houses, because trees do not extrapolate in the same way as a linear equation.

We filled missing values and one-hot encoded categories in a pipeline. We compared different fractions of features considered at each split, leaf sizes, tree depths, and a 20-feature subset. The all-feature setting with max_features=0.3 and min_samples_leaf=1 had **29,478.94** mean RMSE on two checked five-fold partitions in the recorded experiment. The 20-feature version with max_features=0.6 averaged **29,702.21** on three partitions. Later checks with 400 trees also compared max_features=0.2, 0.3, and 0.4; 0.3 was strongest of those three (about **29,651** mean RMSE). These checks had different tree counts, so their exact RMSE values are not interchangeable.

We trained 800 trees with max_features=0.3 and min_samples_leaf=1 on all labeled rows. Kaggle public RMSE was **30,697.63354**. This is useful but not the best.

4. Switching to Ridge and adding simple features

We had used Ridge in our regression quiz. It is still a linear model, but regularization helps keep coefficients from becoming too large when predictors overlap.

We used the supplied numeric and categorical columns (excluding Id and SalePrice), median imputation, a Missing category, one-hot encoding, standardization, and a pipeline. We added straightforward calculations: total area, weighted bathroom count, and house age. A Ridge with alpha=10 and these features averaged **29,300.47** RMSE across three five-fold partitions. Adding OverallQual multiplied by GrLivArea improved that mean to **29,202.54**. Larger Ridge penalties, including alpha=100 and alpha=300, were worse in this comparison.

We kept Ridge and the quality-by-living-area interaction for the next experiments. These early Ridge variants were local checks; no valid public Kaggle score is recorded for them.

5. Capping unusually large living areas

A linear model can predict extremely high prices for houses larger. We tested a few caps on GrLivArea without removing any of the 1,200 training rows.

In a later Ridge setup, no cap gave about **29,206.6** mean RMSE across three five-fold partitions. A cap at 3,500 square feet with alpha=20 gave about **28,670.2**. A cap at 4,000 with alpha=10 gave **28,327.59**. These settings were selected using training labels only.

The 4,000-square-foot capped Ridge scored **54,417.66090** on Kaggle. This was much worse than its local estimate. We could see an extreme test home and a large model prediction, but we could not see its hidden sale price or prove which individual row caused the public error. This result showed that local cross-validation alone was not a sufficient decision rule.

6. Adding the LargeEdwards indicator

During exploratory analysis, very large homes showed different prices across neighborhoods. We tested a neighborhood-and-area indicator so Ridge could represent a possible difference for very large Edwards homes rather than relying only on one global size trend.

LargeEdwards equals 1 only when Neighborhood is Edwards and original GrLivArea exceeds 4,000; otherwise it is 0. It uses no house ID or hidden price. We kept the 4,000-square-foot cap and Ridge(alpha=10). Three five-fold partitions averaged **27,398.92** RMSE, compared with **28,327.59** for the earlier capped model on those partitions.

Kaggle public RMSE improved to **27,643.51679**. We kept this as the best confirmed course-method result at that stage. **Limitation:** exactly one training row has LargeEdwards=1, so the feature is fragile and could overfit the public portion.

7. Averaging Ridge and random forest predictions

Ridge was stronger on the actual public leaderboard, but the forest might correct some of its linear extrapolation errors. We gave Ridge most of the weight.

On the same three five-fold partitions, the 75% Ridge / 25% forest average had **26,728.83** mean RMSE, versus **27,398.92** for Ridge alone and **29,632.94** for forest alone. Other weights (50/50 and 85/15) were weaker than 75/25 locally. Both component estimators were course methods.

The average scored **27,802.62697** on Kaggle, worse than the single Ridge's **27,643.51679**. We rejected it. So further fine-tuning weights to the public score offered only a tiny possible gain and would risk overfitting that public subset.

8. Other changes we tested locally

These checks used the successful single Ridge model as the reference. We tested them to find a small change with a clear housing interpretation.

**Ordered quality encoding.** We replaced one-hot encoding for three, five, or eight quality-rating columns with ordinal encoding. On three five-fold partitions, the mean RMSE values were **28,079.0**, **27,988.2**, and **27,951.8**, respectively, versus **27,398.9** for the one-hot reference. We kept one-hot encoding.

**Small numeric interactions.** The first three-partition check used the same reference Ridge mean RMSE of **27,398.9**:

| Extra feature or features | Mean local RMSE |
| --- | ---: |
| OverallQual × TotalArea | 27,483.8 |
| OverallQual squared | 27,325.8 |
| OverallQual × HouseAge | 27,372.3 |
| OverallQual × GarageArea | 27,270.2 |
| OverallQual × GarageCars | 27,242.0 |
| GarageArea interaction plus quality squared | 27,241.9 |
| GarageArea interaction plus age interaction | 27,273.7 |
| GarageArea interaction plus GarageCars interaction | 27,223.5 |
| GarageArea interaction plus TotalArea interaction | 27,406.1 |
| TotalArea interaction plus quality squared | 27,478.4 |

A two-garage-interaction version was slightly better than the single GarageCars interaction locally, but added another coefficient for only a small extra gain. We chose the **one-feature** QualityGarageCars version for simplicity and checked it on two further partitions. Comparing several candidates on the same folds makes these development scores optimistic.

9. Choosing our final Ridge model

QualityGarageCars = OverallQual × GarageCars has a direct interpretation: the effect of an additional garage space may differ for houses of different overall quality. It is a small use of the polynomial-feature idea taught in class, inside the same Ridge model. It preserves the other preprocessing, the living-area cap, and LargeEdwards.

Across five five-fold partitions, the final model's mean out-of-fold RMSE was **27,184.01**, versus **27,347.14** for the previous Ridge with LargeEdwards on those exact partitions. The final feature improved every checked partition. The average gain was modest, about 163 RMSE, so an actual Kaggle check was necessary.

The new submission scored **27,003.58788**, improving on the previous course-method public score **27,643.51679** by **639.92891**. We promoted this single Ridge model to house_price_model.ipynb, model.py, and submission.csv. The earlier scored Ridge notebook and CSV are preserved in archive/ and model_previous.py.

### Final model

The selected model uses all 79 original house predictors except Id, plus six calculated predictors: TotalArea, TotalBathrooms, HouseAge, QualityLivingArea, LargeEdwards, and QualityGarageCars. The model predicts SalePrice in dollars with Ridge(alpha=10). Numeric missing values are filled with training medians and standardized; missing categories are labeled Missing and one-hot encoded. All preprocessing is fitted inside each training fold during validation.